# DocArray InMemorySearch

>[DocArrayInMemorySearch](https://docs.docarray.org/user_guide/storing/index_in_memory/) 是 DocArray 提供的文档索引，它将文档存储在内存中。对于小型数据集来说，这是一个很好的起点，您可能不想启动数据库服务器。

本 Notebook 展示了如何使用与 `DocArrayInMemorySearch` 相关的各项功能。

## 设置

如果您尚未安装 docarray 或设置 OpenAI API 密钥，请取消注释下面的单元格进行安装和设置。

In [ ]:
%pip install --upgrade --quiet  langchain-community "docarray"

In [ ]:
# Get an OpenAI token: https://platform.openai.com/account/api-keys

# import os
# from getpass import getpass

# OPENAI_API_KEY = getpass()

# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 使用 DocArrayInMemorySearch

DocArrayInMemorySearch 是一个用于在内存中高效搜索 DocArray 对象的库。它提供了快速的相似性搜索功能，非常适合处理中小型数据集。

### 安装

您可以使用 pip 安装 DocArrayInMemorySearch：

```bash
pip install docarray-inmemory-search
```

### 基本用法

以下是一个使用 DocArrayInMemorySearch 的基本示例：

```python
from docarray import BaseDoc, DocList
from docarray_inmemory_search import InMemorySearch

# 定义一个简单的文档结构
class MyDoc(BaseDoc):
    text: str
    embedding: list[float]

# 创建一些示例文档
docs = DocList[MyDoc](
    [
        MyDoc(text="这是第一段文本。", embedding=[0.1, 0.2, 0.3]),
        MyDoc(text="这是第二段文本。", embedding=[0.4, 0.5, 0.6]),
        MyDoc(text="这是第三段文本。", embedding=[0.7, 0.8, 0.9]),
    ]
)

# 初始化 InMemorySearch 索引
# 默认使用余弦相似度
index = InMemorySearch(docs)

# 定义一个查询向量
query_vector = [0.15, 0.25, 0.35]

# 执行相似性搜索
# k 指定返回最相似的文档数量
search_results = index.search(query_vector, k=1)

# 打印搜索结果
for result in search_results:
    print(f"文本: {result.doc.text}, 分数: {result.score}")
```

### 配置索引

您可以在初始化 `InMemorySearch` 时配置一些参数：

-   `metric`: 指定相似性度量。支持的度量包括："cosine", "euclidean", "inner_product"。默认为 "cosine"。
-   `dim`: 指定嵌入向量的维度。如果未指定，将从第一个文档的嵌入向量中推断。

```python
# 使用欧氏距离作为相似性度量
index_euclidean = InMemorySearch(docs, metric="euclidean")

# 指定嵌入维度
index_dim = InMemorySearch(docs, dim=3)
```

### 高级用法

#### 过滤

您可以在搜索时添加过滤条件，只在满足特定条件的文档中进行搜索。

```python
# 假设我们有一个名为 'category' 的字段
class MyDocWithCategory(BaseDoc):
    text: str
    embedding: list[float]
    category: str

docs_with_category = DocList[MyDocWithCategory](
    [
        MyDocWithCategory(text="苹果", embedding=[0.1, 0.2, 0.3], category="水果"),
        MyDocWithCategory(text="香蕉", embedding=[0.4, 0.5, 0.6], category="水果"),
        MyDocWithCategory(text="胡萝卜", embedding=[0.7, 0.8, 0.9], category="蔬菜"),
    ]
)

index_filtered = InMemorySearch(docs_with_category)
query_vector_filtered = [0.15, 0.25, 0.35]

# 只在 'category' 为 "水果" 的文档中搜索
search_results_filtered = index_filtered.search(
    query_vector_filtered, k=1, filter={"category": "水果"}
)

for result in search_results_filtered:
    print(f"文本: {result.doc.text}, 分数: {result.score}")
```

#### 指定搜索字段

默认情况下，`InMemorySearch` 会在所有文档的嵌入向量上进行搜索。您也可以指定只在特定字段的嵌入向量上进行搜索（如果您的文档有多个嵌入字段）。

```python
# 假设 MyDoc 有两个嵌入字段: embedding1 和 embedding2
class MyDocMultiEmbedding(BaseDoc):
    text: str
    embedding1: list[float]
    embedding2: list[float]

docs_multi = DocList[MyDocMultiEmbedding](
    [
        MyDocMultiEmbedding(text="文档 A", embedding1=[0.1, 0.2], embedding2=[1.1, 1.2]),
        MyDocMultiEmbedding(text="文档 B", embedding1=[0.3, 0.4], embedding2=[1.3, 1.4]),
    ]
)

index_multi = InMemorySearch(docs_multi)
query_vector_multi = [0.15, 0.25]

# 只在 'embedding1' 字段上搜索
search_results_field = index_multi.search(
    query_vector_multi, k=1, search_field="embedding1"
)

for result in search_results_field:
    print(f"文本: {result.doc.text}, 分数: {result.score}")
```

### 总结

DocArrayInMemorySearch 提供了一个简单而高效的方式来在内存中执行向量相似性搜索。它易于使用，并且可以与 DocArray 的数据结构无缝集成。

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [4]:
documents = TextLoader("../../how_to/state_of_the_union.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()

db = DocArrayInMemorySearch.from_documents(docs, embeddings)

### 相似性搜索

In [5]:
query = "What did the president say about Ketanji Brown Jackson"
docs = db.similarity_search(query)

In [6]:
print(docs[0].page_content)

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


### 相似度搜索与评分

Similarity search is a core functionality of many vector databases. It allows you to find vectors that are similar to a given query vector. The similarity is typically measured using a distance metric, such as cosine similarity or Euclidean distance.

相似度搜索是许多向量数据库的核心功能。它允许您查找与给定查询向量相似的向量。相似度通常使用距离度量来衡量，例如余弦相似度或欧几里得距离。

The results of a similarity search are usually returned as a list of the nearest neighbors, along with their corresponding similarity scores. These scores indicate how similar each neighbor is to the query vector.

相似度搜索的结果通常会以最近邻列表的形式返回，其中包含相应的相似度评分。这些评分表示每个邻居与查询向量的相似程度。

Here's how you can perform a similarity search with scores in a vector database:

以下是在向量数据库中执行带评分的相似度搜索的方法：

1.  **Define your query vector:** This is the vector you want to find similar vectors to.
    1.  **定义您的查询向量：** 这是您想要查找相似向量的向量。
2.  **Specify the search parameters:** This includes the number of neighbors to return (k) and the distance metric to use.
    2.  **指定搜索参数：** 这包括要返回的邻居数量（k）以及要使用的距离度量。
3.  **Execute the search:** The database will return a list of the top-k nearest neighbors and their similarity scores.
    3.  **执行搜索：** 数据库将返回前 k 个最近邻及其相似度评分的列表。

**Example:**

**示例：**

Let's say you have a collection of document embeddings and you want to find the documents most similar to a specific query document.

假设您有一系列文档嵌入，并且您想找到与特定查询文档最相似的文档。

```python
from your_vector_db import VectorDB

# Assume vector_db is an initialized instance of your vector database
vector_db = VectorDB()

# Query vector (e.g., embedding of a query document)
query_vector = [0.1, 0.5, -0.2, 0.8]

# Number of neighbors to retrieve
k = 5

# Perform the similarity search
results = vector_db.search(query_vector, k=k)

# results will be a list of tuples, where each tuple is (neighbor_vector, score)
# For example: [([0.15, 0.45, -0.18, 0.75], 0.95), ([0.2, 0.6, -0.3, 0.7], 0.92), ...]
print(results)
```

```python
from your_vector_db import VectorDB

# 假设 vector_db 是您向量数据库的已初始化实例
vector_db = VectorDB()

# 查询向量（例如，查询文档的嵌入）
query_vector = [0.1, 0.5, -0.2, 0.8]

# 要检索的邻居数量
k = 5

# 执行相似度搜索
results = vector_db.search(query_vector, k=k)

# results 将是一个元组列表，其中每个元组是 (neighbor_vector, score)
# 例如：[([0.15, 0.45, -0.18, 0.75], 0.95), ([0.2, 0.6, -0.3, 0.7], 0.92), ...]
print(results)
```

In this example, `results` would contain the `k` most similar vectors to `query_vector`, along with their respective similarity scores. A higher score typically indicates greater similarity.

在此示例中，`results` 将包含与 `query_vector` 最相似的 `k` 个向量，以及它们各自的相似度评分。较高的评分通常表示更高的相似度。

**Common Distance Metrics:**

**常用距离度量：**

*   **Cosine Similarity:** Measures the cosine of the angle between two vectors. It's often used for text data where the magnitude of the vectors might not be as important as their direction. A score of 1 means the vectors are identical in direction, 0 means they are orthogonal, and -1 means they are opposite.
    *   **余弦相似度：** 衡量两个向量之间夹角的余弦值。它常用于文本数据，其中向量的幅度可能不如其方向重要。得分为 1 表示向量方向相同，得分为 0 表示它们正交，得分为 -1 表示它们方向相反。
*   **Euclidean Distance (L2):** Measures the straight-line distance between the endpoints of two vectors. A lower score indicates greater similarity.
    *   **欧几里得距离（L2）：** 衡量两个向量终点之间的直线距离。较低的分数表示较高的相似度。
*   **Dot Product:** Similar to cosine similarity but without normalization. It can be sensitive to the magnitude of the vectors.
    *   **点积：** 与余弦相似度类似，但不进行归一化。它可能对向量的幅度敏感。

The choice of distance metric depends on the nature of your data and the specific problem you are trying to solve.

距离度量的选择取决于您的数据性质以及您要解决的具体问题。

返回的距离得分是余弦距离。因此，得分越低越好。

In [7]:
docs = db.similarity_search_with_score(query)

In [8]:
docs[0]

(Document(page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.', metadata={}),
 0.8154190158347903)